# AndinaLog 03B · Notebook 1 · Diagnóstico IoT didáctico v2

Se conserva Bronze y se añade, por cada columna, `*_en_cuarentena` y `*_motivo`.
`en_cuarentena` es el OR de las banderas por columna. `False` no significa que
toda regla contextual se haya podido evaluar. La hora sin zona es de Bolivia.
Fahrenheit no cumple la unidad de ingreso C y pasa a cuarentena diagnóstica;
el tratamiento puede convertirlo y recuperarlo para Silver.


In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
ZONA_HORARIA_ORIGEN = "America/La_Paz"
COLUMNAS_BRONZE = [
    "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag",
]

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "datasets/AndinaLog_03B_Bronce/andinalog_iot_telemetry.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets/AndinaLog_03B_Bronce/andinalog_iot_telemetry.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
RUTA_BRONZE = RAIZ / "datasets/AndinaLog_03B_Bronce/andinalog_iot_telemetry.csv"
RUTA_PRODUCTOS = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_productos/salidas/andinalog_productos_didactico_v1_silver.csv"
RUTA_FLOTA = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_flota/salidas/andinalog_flota_didactico_v1_silver.csv"
SALIDAS = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_iot_telemetry/salidas"
HASH_BRONZE = hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()

bronze = pd.read_csv(RUTA_BRONZE, dtype="string", encoding="utf-8-sig", keep_default_na=False)
if list(bronze.columns) != COLUMNAS_BRONZE:
    raise ValueError(f"Esquema Bronze inesperado: {list(bronze.columns)}")
df = bronze.copy(deep=True)
df.insert(0, "fila_bronze", range(1, len(df)+1))
print("Lecturas Bronze:", len(df))


## Reglas visibles de diagnóstico

Las reglas internas acumulan motivos por campo. La salida conserva solo una
bandera `*_en_cuarentena` y un `*_motivo` por columna. Un motivo puede explicar
una observación que no obliga a cuarentena; `False` no significa verificación
universal. El dato Bronze nunca se modifica.


In [ ]:
# Las reglas están visibles aquí, sin depender de un catálogo externo.
PRIORIDAD = {"OK":0, "NO_EVALUABLE":1, "REVISAR":2, "CRITICO":3}
for columna in COLUMNAS_BRONZE:
    df[f"{columna}_estado"] = "OK"
    df[f"{columna}_motivo"] = ""

def marcar(columna, mascara, estado, motivo):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    e, m = f"{columna}_estado", f"{columna}_motivo"
    subir = mascara & df[e].map(PRIORIDAD).lt(PRIORIDAD[estado])
    df.loc[subir, e] = estado
    previo = df.loc[mascara, m]
    df.loc[mascara, m] = previo.where(previo.eq(""), previo + "; ") + motivo

def no_vacio(columna):
    return df[columna].str.strip().ne("")

# 1. Completitud y formato de fecha e identificadores.
for columna in COLUMNAS_BRONZE:
    marcar(columna, ~no_vacio(columna), "CRITICO", "Valor faltante")

fecha = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
forma_fecha = df["timestamp"].str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}").fillna(False)
marcar("timestamp", no_vacio("timestamp") & (~forma_fecha | fecha.isna()), "CRITICO", "Formato o fecha de calendario inválidos")

for columna, patron in {
    "viaje_id":r"VIA-\d{5}", "order_id":r"ORD-\d{4}-\d{5}",
    "camion_id":r"CAM-\d{2}", "producto_id":r"PROD-\d{3}",
}.items():
    coincide = df[columna].str.fullmatch(patron).fillna(False)
    marcar(columna, no_vacio(columna) & ~coincide, "CRITICO", f"Formato esperado: {patron}")

# 2. Magnitudes y unidades según el dominio de cadena de frío.
temperatura = pd.to_numeric(df["temperatura_cabina_c"], errors="coerce")
humedad = pd.to_numeric(df["humedad_cabina_pct"], errors="coerce")
marcar("temperatura_cabina_c", no_vacio("temperatura_cabina_c") & temperatura.isna(), "CRITICO", "Temperatura no numérica")
marcar("temperatura_cabina_c", temperatura.eq(-999), "CRITICO", "-999 es centinela: no es una temperatura")
marcar("humedad_cabina_pct", no_vacio("humedad_cabina_pct") & humedad.isna(), "CRITICO", "Humedad no numérica")
marcar("humedad_cabina_pct", humedad.notna() & ~humedad.between(0,100), "CRITICO", "Humedad fuera de 0 a 100 %")

unidad = df["temp_unit"].str.strip().str.upper()
marcar("temp_unit", unidad.eq("F"), "CRITICO", "Fahrenheit no cumple unidad C de ingesta; convertir en tratamiento")
marcar("temp_unit", unidad.eq("K"), "CRITICO", "Kelvin no es unidad operacional esperada en AndinaLog")
marcar("temp_unit", no_vacio("temp_unit") & ~unidad.isin(["C","F","K"]), "CRITICO", "Unidad de temperatura desconocida")
marcar("temperatura_cabina_c", unidad.eq("K") & temperatura.notna() & temperatura.ne(-999), "NO_EVALUABLE", "Rango térmico no evaluado con unidad K")

for columna in ("desviacion_termica_flag", "desviacion_proximos_60min_flag"):
    marcar(columna, no_vacio(columna) & ~df[columna].isin(["0","1"]), "CRITICO", "Bandera distinta de 0 o 1")

# 3. Clave candidata: una lectura por viaje y timestamp.
firma = pd.util.hash_pandas_object(df[COLUMNAS_BRONZE], index=False)
variantes = firma.groupby([df["viaje_id"], df["timestamp"]], dropna=False).transform("nunique")
clave_repetida = df.duplicated(["viaje_id","timestamp"], keep=False)
copia = clave_repetida & variantes.eq(1) & df.duplicated(COLUMNAS_BRONZE, keep="first")
conflicto = clave_repetida & variantes.gt(1)
for columna in ("viaje_id", "timestamp"):
    marcar(columna, copia, "CRITICO", "Copia idéntica de la lectura; no contar dos veces")
    marcar(columna, conflicto, "CRITICO", "Misma clave con datos diferentes; revisar ambas lecturas")

# 4. Cobertura parcial de dimensiones Silver; ausencia es advertencia, no prueba de ID inválido.
productos = None
if RUTA_PRODUCTOS.is_file():
    productos = pd.read_csv(RUTA_PRODUCTOS, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    if productos["producto_id_tratado"].duplicated().any():
        raise ValueError("Productos Silver tiene producto_id_tratado duplicado")
    marcar("producto_id", no_vacio("producto_id") & ~df["producto_id"].isin(productos["producto_id_tratado"]),
           "REVISAR", "Sin correspondencia en Productos Silver de cobertura parcial")

if RUTA_FLOTA.is_file():
    flota = pd.read_csv(RUTA_FLOTA, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    if flota["camion_id_tratado"].duplicated().any():
        raise ValueError("Flota Silver tiene camion_id_tratado duplicado")
    camion_formato_valido = df["camion_id"].str.fullmatch(r"CAM-\d{2}").fillna(False)
    marcar("camion_id", camion_formato_valido & ~df["camion_id"].str.strip().str.upper().isin(flota["camion_id_tratado"]),
           "REVISAR", "Sin correspondencia en Flota Silver de cobertura parcial")

# 5. Coherencia entre bandera térmica y rango del producto. Una excursión correcta no es error.
if productos is not None:
    p = productos.set_index("producto_id_tratado")
    objetivo = pd.to_numeric(df["producto_id"].map(p["temperatura_conservacion_requerida_c_tratada"]), errors="coerce")
    tolerancia = pd.to_numeric(df["producto_id"].map(p["tolerancia_temperatura_c_tratada"]), errors="coerce")
    temp_c = temperatura.where(unidad.eq("C"), (temperatura-32)*5/9)
    evaluable = (unidad.isin(["C","F"]) & temperatura.notna() & temperatura.ne(-999)
                 & objetivo.notna() & tolerancia.notna() & tolerancia.ge(0)
                 & df["desviacion_termica_flag"].isin(["0","1"]))
    fuera = (temp_c-objetivo).abs() > tolerancia
    incoherente = evaluable & fuera.ne(df["desviacion_termica_flag"].eq("1"))
    marcar("desviacion_termica_flag", incoherente, "REVISAR", "Flag no coincide con objetivo y tolerancia del producto")

print("Reglas aplicadas; no se modificaron columnas Bronze")


## Banderas por columna y decisión por fila

`en_cuarentena` es verdadero si alguna bandera por columna es verdadera. Los
motivos de cada campo evitan un resumen textual duplicado por fila.


In [ ]:
# Los estados son internos para aplicar reglas; la salida didáctica usa banderas y motivos.
for columna in COLUMNAS_BRONZE:
    df[f"{columna}_en_cuarentena"] = df[f"{columna}_estado"].eq("CRITICO")
df["en_cuarentena"] = df[[f"{c}_en_cuarentena" for c in COLUMNAS_BRONZE]].any(axis=1)
df["zona_horaria_origen"] = ZONA_HORARIA_ORIGEN
salida_columnas = (["fila_bronze", *COLUMNAS_BRONZE] +
    [item for c in COLUMNAS_BRONZE for item in (f"{c}_en_cuarentena", f"{c}_motivo")] +
    ["en_cuarentena", "zona_horaria_origen"])
diagnosticado = df[salida_columnas].copy()
cuarentena = diagnosticado.loc[diagnosticado["en_cuarentena"]].copy()

pd.testing.assert_frame_equal(diagnosticado[COLUMNAS_BRONZE], bronze)
assert len(diagnosticado) == len(bronze)
assert diagnosticado["fila_bronze"].is_unique
assert cuarentena["en_cuarentena"].all()
assert len(cuarentena) == int(diagnosticado["en_cuarentena"].sum())
assert cuarentena.apply(lambda fila: any(
    fila[f"{c}_en_cuarentena"] and bool(fila[f"{c}_motivo"])
    for c in COLUMNAS_BRONZE), axis=1).all()
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest() == HASH_BRONZE
print("Diagnosticadas:", len(diagnosticado), "| Cuarentena diagnóstica:", len(cuarentena))
display(diagnosticado[["fila_bronze", "temp_unit", "temp_unit_en_cuarentena",
    "temp_unit_motivo", "en_cuarentena"]].head(10))


## Exportación

Se generan tres CSV: diagnosticado completo, subconjunto de cuarentena y
reporte de calidad. El Bronze permanece intacto.


In [ ]:
# Tres salidas: completo, subconjunto de cuarentena y reporte de calidad.
SALIDAS.mkdir(parents=True, exist_ok=True)
ruta_diagnosticado = SALIDAS / "andinalog_iot_telemetry_didactico_v2_diagnosticado.csv"
ruta_cuarentena = SALIDAS / "andinalog_iot_telemetry_didactico_v2_cuarentena_diagnostico.csv"
ruta_reporte = SALIDAS / "andinalog_iot_telemetry_didactico_v2_reporte_calidad_diagnostico.csv"
reporte = pd.DataFrame([
    ("filas_bronze", len(bronze)),
    ("filas_diagnosticadas", len(diagnosticado)),
    ("filas_cuarentena_diagnostico", len(cuarentena)),
    ("unidades_fahrenheit", int(unidad.eq("F").sum())),
    ("unidades_kelvin", int(unidad.eq("K").sum())),
    ("temperaturas_centinela", int(temperatura.eq(-999).sum())),
    ("fechas_invalidas", int(fecha.isna().sum())),
    ("copias_exactas", int(copia.sum())),
    ("claves_conflictivas", int(conflicto.sum())),
], columns=["metrica", "valor"])
diagnosticado.to_csv(ruta_diagnosticado, index=False, encoding="utf-8-sig")
cuarentena.to_csv(ruta_cuarentena, index=False, encoding="utf-8-sig")
reporte.to_csv(ruta_reporte, index=False, encoding="utf-8-sig")
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest() == HASH_BRONZE
display(reporte)
print(ruta_diagnosticado, ruta_cuarentena, ruta_reporte, sep="\n")


## Interpretación y siguiente etapa

El diagnosticado contiene todas las lecturas. La cuarentena diagnóstica es un
subconjunto del diagnosticado y no debe concatenarse de nuevo como entrada.
El tratamiento lee el diagnosticado completo y decide qué filas se recuperan.
Los motivos por columna explican las decisiones sin una columna de motivos
repetidos por fila. El reporte presenta los conteos de este lote.
